In [2]:
# 트레인 이미지 흑백화

import os
import cv2
import copy
# 원본 이미지 폴더
input_folder = "pill_train/before"

# 저장할 폴더
output_folder = "pill_train/after"

# 저장 폴더가 없으면 생성
os.makedirs(output_folder, exist_ok=True)

# 처리할 확장자
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")

for filename in os.listdir(input_folder):
    if filename.lower().endswith(extensions):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        # 이미지 읽기
        img = cv2.imread(input_path)

        if img is None:
            print(f"읽기 실패: {filename}")
            continue

        # 흑백 변환
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # 저장
        cv2.imwrite(output_path, gray)

        print(f"완료: {filename}")

print("모든 이미지 변환 완료!")

완료: WIN_20260729_16_17_32_Pro.jpg
완료: WIN_20260729_16_17_34_Pro.jpg
완료: WIN_20260729_16_17_35_Pro.jpg
완료: WIN_20260729_16_17_37_Pro.jpg
완료: WIN_20260729_16_17_39_Pro.jpg
완료: WIN_20260729_16_17_40_Pro.jpg
완료: WIN_20260729_16_17_41_Pro.jpg
완료: WIN_20260729_16_17_43_Pro.jpg
완료: WIN_20260729_16_17_48_Pro.jpg
완료: WIN_20260729_16_17_51_Pro.jpg
완료: WIN_20260729_16_17_52_Pro.jpg
완료: WIN_20260729_16_17_56_Pro.jpg
완료: WIN_20260729_16_17_59_Pro.jpg
완료: WIN_20260729_16_18_01_Pro.jpg
완료: WIN_20260729_16_18_03_Pro.jpg
완료: WIN_20260729_16_18_05_Pro.jpg
완료: WIN_20260729_16_18_10_Pro.jpg
완료: WIN_20260729_16_18_11_Pro.jpg
완료: WIN_20260729_16_18_13_Pro.jpg
완료: WIN_20260729_16_18_14_Pro.jpg
완료: WIN_20260729_16_18_15_Pro.jpg
완료: WIN_20260729_16_18_16_Pro.jpg
완료: WIN_20260729_16_18_18_Pro.jpg
완료: WIN_20260729_16_18_19_Pro.jpg
완료: WIN_20260729_16_18_23_Pro.jpg
완료: WIN_20260729_16_18_25_Pro.jpg
완료: WIN_20260729_16_18_26_Pro.jpg
완료: WIN_20260729_16_18_27_Pro.jpg
완료: WIN_20260729_16_18_28_Pro.jpg
완료: WIN_202607

In [3]:
# PIP 설치 PySide / pyinstaller

# !pip3 install PySide6
# !pip3 install pyinstaller

# !pip uninstall opencv-python opencv-contrib-python opencv-python-headless

In [4]:
# yaml 수정

import yaml
import os
data = {}

with open("pill_train/train_set/data.yaml", "r") as file:
    data = yaml.safe_load(file)
    data["path"] = os.path.abspath("pill_train/train_set")
    data["train"] = "train/images"
    data["val"] = "valid/images"

with open("pill_train/train_set/data.yaml", "w") as file:
    yaml.dump(data, file)

In [ ]:
# 촬영 데이터로 이미지 학습

from ultralytics import YOLO
if __name__ == "__main__":
    model = YOLO("yolov8s.pt")

    model.train(
        data="pill_train/train_set/data.yaml",
        epochs=50,
        imgsz=640,
        batch=32,
        optimizer="AdamW",
        lr0=0.001
    )

New https://pypi.org/project/ultralytics/8.4.115 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.107  Python-3.9.23 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pill_train/train_set/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0,

KeyboardInterrupt: 

In [ ]:
# 학습 모델을 이용한 판정 테스트 코드

import cv2

from ultralytics import YOLO

colors = {
    0: (0, 255, 0),    # class 0 → 초록
    1: (0, 0, 255),    # class 1 → 빨강
    2: (255, 0, 0),    # class 2 → 파랑
}

model = YOLO("pill_retrain.pt")

cap = cv2.VideoCapture(0)
if cap.isOpened():
    while True:
        ret, frame = cap.read()
        if not ret:
            print("error")
            break
        dark = cv2.convertScaleAbs(frame, alpha=0.9, beta=-40)
        gray_frame = cv2.cvtColor(dark, cv2.COLOR_BGR2GRAY)

        results = model(gray_frame, conf=0.7)

        for result in results:
            for box in result.boxes:
                # 좌표
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # confidence
                conf = float(box.conf[0])

                # class
                cls = int(box.cls[0])
                name = model.names[cls]
                color = colors.get(cls, (255, 255, 255))

                # 원본 frame에 박스 표시
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    color,
                    2
                )

                cv2.putText(
                    frame,
                    f"{name} {conf:.2f}",
                    (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    color,
                    2
                )
            cv2.imshow("FRAME",frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cv2.destroyAllWindows()
cv2.waitKey(1)



0: 480x640 (no detections), 7.6ms
Speed: 2.2ms preprocess, 7.6ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 5.9ms
Speed: 1.0ms preprocess, 5.9ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 5.9ms
Speed: 0.9ms preprocess, 5.9ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 5.2ms
Speed: 0.9ms preprocess, 5.2ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 4.7ms
Speed: 1.1ms preprocess, 4.7ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 pill_c, 6.8ms
Speed: 1.0ms preprocess, 6.8ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 4.5ms
Speed: 1.0ms preprocess, 4.5ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 5.2ms
Speed: 1.0ms preprocess, 5.2ms inference, 0.5ms postpro

-1

In [ ]:
# 웹캠 UI 예시

import sys
import cv2
from PySide6.QtWidgets import (QApplication, QWidget, 
                               QLabel, QPushButton,
                               QVBoxLayout, QHBoxLayout, QTextEdit)
from PySide6.QtCore import QTimer, Qt
from PySide6.QtGui import QImage, QPixmap
from ultralytics import YOLO
import copy

colors = {
    0: (0, 255, 0),    # class 0 → 초록
    1: (0, 0, 255),    # class 1 → 빨강
    2: (255, 0, 0),    # class 2 → 파랑
}

class My_App(QWidget):
    def __init__(self):
        super().__init__()
        
        # 웹캠 관련 변수
        self.cap = None
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update_frame) # 타이머가 울릴 때마다 프레임 갱신
        self.pill_type = ['a','b','c']
        self.colors = {
            0: (0, 255, 0),
            1: (0, 0, 255),
            2: (255, 0, 0),
        }
        self.model = YOLO("pill_retrain.pt")
        self.pill_class = []
        self.edtPdaily = []
        self.lbTotalCount = []
        self.lbLeftDay = []
        self.init_UI()
        
    def init_UI(self):
        self.setWindowTitle("실시간 웹캠 플레이어")
        self.resize(1200, 560)

        # 1. 웹캠 화면을 출력할 QLabel 생성
        self.image_label = QLabel("", self)
        self.image_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.image_label.setStyleSheet("background-color: #222; color: #fff; font-size: 16px;")
        self.image_label.setMinimumSize(640, 480)

        if self.cap is None or not self.cap.isOpened():
            self.cap = cv2.VideoCapture(0) # 0번 기본 웹캠 열기
                    
            if not self.timer.isActive():
                self.timer.start(30) # 30ms 간격으로 update_frame 호출 (약 33 FPS)
        # 2. 시작 / 일시정지 버튼 생성
        self.btn_start = QPushButton("시작", self)

        # 버튼 스타일링 (선택 사항)
        self.btn_start.setFixedHeight(40)

        # 버튼 클릭 이벤트 연결
        self.btn_start.clicked.connect(self.start_webcam)

        self.pill_layout_list = []

        for i in range(3):
            pill_child_v = QVBoxLayout()

            pill_child_h = QHBoxLayout() 
            label = QLabel(f'{self.pill_type[i]} 타입 하루섭취량')
            self.textedit = QTextEdit()
            pill_child_h.addWidget(label)
            pill_child_h.addWidget(self.textedit)
            self.edtPdaily.append(self.textedit)
            pill_child_v.addLayout(pill_child_h)

            pill_child_h = QHBoxLayout() 
            label = QLabel(f'{self.pill_type[i]} 타입 총개수')
            pill_child_h.addWidget(label)
            self.label = QLabel('')
            pill_child_h.addWidget(self.label)
            self.lbTotalCount.append(self.label)
            pill_child_v.addLayout(pill_child_h)

            pill_child_h = QHBoxLayout() 
            label = QLabel(f'{self.pill_type[i]} 타입 몇일분')
            pill_child_h.addWidget(label)
            self.label = QLabel('')
            pill_child_h.addWidget(self.label)
            self.lbLeftDay.append(self.label)
            pill_child_v.addLayout(pill_child_h)

            self.pill_layout_list.append(pill_child_v)

        # 왼쪽 수직 레이아웃 (VBox)
        left_layout = QVBoxLayout()
        left_layout.addWidget(self.image_label)     # 상단: 웹캠 화면
        left_layout.addWidget(self.btn_start)        # 하단: 버튼 레이아웃

        # 오른쪽 수직 레이아웃 (VBox)
        right_layout = QVBoxLayout()
        right_layout.setAlignment(Qt.AlignmentFlag.AlignCenter)

        for i, self.layout in enumerate(self.pill_layout_list):
            right_layout.addLayout(self.layout)
        

        main_layout = QHBoxLayout()
        main_layout.addLayout(left_layout)
        main_layout.addLayout(right_layout)
    
        self.setLayout(main_layout)

    def start_webcam(self):
        temp = copy.deepcopy(self.pill_class)
        for i in range(3):
            pNum = 0
            edt:QTextEdit = self.edtPdaily[i]
            if '' != edt.toPlainText():
                pNum=int(edt.toPlainText())
            
            self.label_temp:QLabel = self.lbTotalCount[i]
            self.label_temp.setText(f"{temp[i]} 개")

            if pNum != 0:
                self.label_temp:QLabel = self.lbLeftDay[i]
                pNum = temp[i] / pNum
                self.label_temp.setText(f"{pNum} 일 분")



    def update_frame(self):
        """OpenCV 프레임을 읽어서 PySide QLabel에 그리기"""
        self.pill_class = [0,0,0]
        if self.cap and self.cap.isOpened():
            ret, frame = self.cap.read()
            if ret:
                # 1. OpenCV(BGR) -> RGB 변환

                dark = cv2.convertScaleAbs(frame, alpha=0.9, beta=-40)
                gray_frame = cv2.cvtColor(dark, cv2.COLOR_BGR2GRAY)
                
                results = self.model(gray_frame, conf=0.7, verbose=False)
                for result in results:
                    for box in result.boxes:
                        # 좌표
                        x1, y1, x2, y2 = map(int, box.xyxy[0])

                        # confidence
                        conf = float(box.conf[0])
        
                        # class
                        cls = int(box.cls[0])
                        if 0 <= cls <= 2:
                            self.pill_class[cls] +=1
                        name = self.model.names[cls]
                        color = colors.get(cls, (255, 255, 255))
        
                        # 원본 frame에 박스 표시
                        cv2.rectangle(
                            frame,
                            (x1, y1),
                            (x2, y2),
                            color,
                            2
                        )
        
                        cv2.putText(
                            frame,
                            f"{name} {conf:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.7,
                            color,
                            2
                        )
                rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

                # 2. QImage 변환
                h, w, ch = rgb_image.shape
                bytes_per_line = ch * w
                qt_image = QImage(rgb_image.data, w, h, bytes_per_line, QImage.Format.Format_RGB888)
                # 3. QLabel 크기에 맞게 Scaled Pixmap 생성 후 적용
                pixmap = QPixmap.fromImage(qt_image)
                scaled_pixmap = pixmap.scaled(
                    self.image_label.size(), 
                    Qt.AspectRatioMode.KeepAspectRatio, 
                    Qt.TransformationMode.SmoothTransformation
                )
                self.image_label.setPixmap(scaled_pixmap)

    def closeEvent(self, event):
        """프로그램 종료 시 웹캠 자원 해제"""
        self.timer.stop()
        if self.cap and self.cap.isOpened():
            self.cap.release()
        event.accept()

if __name__ == "__main__":
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    exam = My_App()
    exam.show()
    sys.exit(app.exec())

SystemExit: 0

c:\Users\kccistc\.conda\envs\torch_env\lib\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# 사용안하는 코드

from PySide6.QtWidgets import *
from PySide6.QtCore import Qt
import sys
import cv2
from PySide6.QtWidgets import (QApplication, QWidget, 
                               QLabel, QPushButton,
                               QVBoxLayout, QHBoxLayout, QTextEdit)
from PySide6.QtCore import QTimer, Qt
from PySide6.QtGui import QImage, QPixmap
from ultralytics import YOLO
import copy

class Demo(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("알약 재고 관리 시스템")
        self.resize(1200,700)
        self.setStyleSheet("""
        QWidget{background:#f4f7fb;font-family:'Malgun Gothic';font-size:11pt;}
        QLabel{color:#333;}
        QGroupBox{
            background:white;
            border:1px solid #d6dde8;
            border-radius:10px;
            margin-top:12px;
            font-weight:bold;
        }
        QGroupBox::title{subcontrol-origin:margin;left:12px;padding:0 4px;}
        QPushButton{
            background:#2f80ed;color:white;border:none;
            border-radius:8px;padding:10px;font-weight:bold;
        }
        QPushButton:hover{background:#1f6fd8;}
        QLineEdit{
            border:1px solid #c5cfdb;
            border-radius:6px;
            padding:4px;
            background:white;
        }
        """)
        main=QHBoxLayout(self)
        left=QVBoxLayout()
        cam=QLabel("Camera Preview")
        cam.setAlignment(Qt.AlignCenter)
        cam.setMinimumSize(640,480)
        cam.setStyleSheet("background:white;border:2px solid #c5cfdb;border-radius:10px;")
        left.addWidget(cam)
        left.addWidget(QPushButton("검사 시작"))

        right=QVBoxLayout()
        for t in ["A","B","C"]:
            g=QGroupBox(f"💊 {t} 타입")
            f=QFormLayout()
            e=QLineEdit()
            e.setAlignment(Qt.AlignCenter)
            e.setFixedWidth(80)
            total=QLabel("0 개")
            total.setStyleSheet("font-weight:bold;color:#2f80ed;")
            days=QLabel("0 일")
            days.setStyleSheet("font-weight:bold;color:#2f80ed;")
            f.addRow("하루 복용량",e)
            f.addRow("현재 개수",total)
            f.addRow("남은 일수",days)
            g.setLayout(f)
            right.addWidget(g)
        main.addLayout(left,2)
        main.addLayout(right,1)
        self.cap = None
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update_frame) # 타이머가 울릴 때마다 프레임 갱신
        self.pill_type = ['a','b','c']
        self.colors = {
            0: (0, 255, 0),
            1: (0, 0, 255),
            2: (255, 0, 0),
        }
        self.model = YOLO("pill_retrain.pt")
        self.pill_class = []
        self.edtPdaily = []
        self.lbTotalCount = []
        self.lbLeftDay = []
        self.init_UI()

    def init_UI(self):
            if self.cap is None or not self.cap.isOpened():
                self.cap = cv2.VideoCapture(0) # 0번 기본 웹캠 열기
                        
                if not self.timer.isActive():
                    self.timer.start(30) # 30ms 간격으로 update_frame 호출 (약 33 FPS)

            self.btn_start.clicked.connect(self.start_webcam)
    
            self.pill_layout_list = []
    
            for i in range(3):
                pill_child_v = QVBoxLayout()
    
                pill_child_h = QHBoxLayout() 
                label = QLabel(f'{self.pill_type[i]} 타입 하루섭취량')
                self.textedit = QTextEdit()
                pill_child_h.addWidget(label)
                pill_child_h.addWidget(self.textedit)
                self.edtPdaily.append(self.textedit)
                pill_child_v.addLayout(pill_child_h)
    
                pill_child_h = QHBoxLayout() 
                label = QLabel(f'{self.pill_type[i]} 타입 총개수')
                pill_child_h.addWidget(label)
                self.label = QLabel('')
                pill_child_h.addWidget(self.label)
                self.lbTotalCount.append(self.label)
                pill_child_v.addLayout(pill_child_h)
    
                pill_child_h = QHBoxLayout() 
                label = QLabel(f'{self.pill_type[i]} 타입 몇일분')
                pill_child_h.addWidget(label)
                self.label = QLabel('')
                pill_child_h.addWidget(self.label)
                self.lbLeftDay.append(self.label)
                pill_child_v.addLayout(pill_child_h)
    
                self.pill_layout_list.append(pill_child_v)
    
            # 왼쪽 수직 레이아웃 (VBox)
            left_layout = QVBoxLayout()
            left_layout.addWidget(self.image_label)     # 상단: 웹캠 화면
            left_layout.addWidget(self.btn_start)        # 하단: 버튼 레이아웃
    
            # 오른쪽 수직 레이아웃 (VBox)
            right_layout = QVBoxLayout()
            right_layout.setAlignment(Qt.AlignmentFlag.AlignCenter)
    
            for i, self.layout in enumerate(self.pill_layout_list):
                right_layout.addLayout(self.layout)
            
    
            main_layout = QHBoxLayout()
            main_layout.addLayout(left_layout)
            main_layout.addLayout(right_layout)
        
            self.setLayout(main_layout)

if __name__=="__main__":
    from PySide6.QtWidgets import QApplication
    app=QApplication([])
    w=Demo()
    w.show()
    app.exec()

RuntimeError: Please destroy the QApplication singleton before creating a new QApplication instance.

In [ ]:
import sys
import cv2
import copy

from PySide6.QtWidgets import (
    QApplication,
    QWidget,
    QLabel,
    QPushButton,
    QVBoxLayout,
    QHBoxLayout,
    QGroupBox,
    QFormLayout,
    QLineEdit,
    QFrame
)
from PySide6.QtGui import QImage, QPixmap
from ultralytics import YOLO

class My_App(QWidget):
    def __init__(self):
        super().__init__()
        # 웹캠 변수
        self.cap = None
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update_frame)
        # 약 타입
        self.pill_type = ['A', 'B', 'C']
        self.colors = {
            0: (0, 255, 0),
            1: (0, 0, 255),
            2: (255, 0, 0),
        }
        # YOLO 모델
        self.model = YOLO("pill_retrain.pt")
        self.pill_class = [0, 0, 0]
        # UI 저장용
        self.edtPdaily = []
        self.lbTotalCount = []
        self.lbLeftDay = []
        self.init_UI()

    def init_UI(self):
        self.setWindowTitle("알약 분류 및 개수 카운팅 시스템")
        self.resize(1280, 700)
        self.setStyleSheet("""
            QWidget {
                background-color: #f7faff;
                font-family: "Malgun Gothic";
                font-size: 14px;
            }
            QLabel {
                color: #333333;
            }
            QGroupBox {
                background-color: white;
                border: 2px solid #d6e7ff;
                border-radius: 15px;
                margin-top: 15px;
                padding: 15px;
                font-size: 16px;
                font-weight: bold;
                color: #1769aa;
            }

            QGroupBox::title {
                subcontrol-origin: margin;
                left: 15px;
                padding: 0 8px;
            }

            QLineEdit {
                background-color: #ffffff;
                border: 1px solid #b8d5ff;
                border-radius: 8px;
                padding: 8px;
                color: #333333;
            }

            QLineEdit:focus {
                border: 2px solid #1976d2;
            }

            QPushButton {
                background-color: #1976d2;
                color: white;
                border-radius: 10px;
                height: 45px;
                font-size: 15px;
                font-weight: bold;
            }

            QPushButton:hover {
                background-color: #1565c0;
            }

            QPushButton:pressed {
                background-color: #0d47a1;
            }
        """)

        # ==========================
        # 웹캠 영역
        # ==========================

        self.image_label = QLabel()
        self.image_label.setAlignment(
            Qt.AlignmentFlag.AlignCenter
        )
        self.image_label.setMinimumSize(
            700,
            520
        )
        self.image_label.setStyleSheet("""
            QLabel {
                background-color: #111827;
                border-radius: 20px;
                border: 4px solid #90caf9;
            }
        """)

        # ==========================
        # 시작 버튼
        # ==========================

        self.btn_start = QPushButton(
            "분석 시작"
        )
        self.btn_start.clicked.connect(
            self.start_webcam
        )

        # ==========================
        # A/B/C 카드 생성
        # ==========================

        self.pill_cards = []

        for i in range(3):
            card = QGroupBox(f"{self.pill_type[i]} Type")
            form = QFormLayout()
            form.setSpacing(12)
            # 하루 섭취량 입력

            daily_edit = QLineEdit()
            daily_edit.setPlaceholderText("하루 복용 개수 입력")
            self.edtPdaily.append(daily_edit)
            form.addRow("하루 섭취량",daily_edit)
            # 총 개수 출력
            total_label = QLabel("0 개")
            total_label.setStyleSheet("""
                QLabel {
                    color:#1565c0;
                    font-size:18px;
                    font-weight:bold;
                }
            """)
            self.lbTotalCount.append(total_label)
            form.addRow("현재 개수",total_label)

            # 남은 날짜 출력
            day_label = QLabel("0 일")
            day_label.setStyleSheet("""
                QLabel {
                    color:#1565c0;
                    font-size:18px;
                    font-weight:bold;
                }
            """)

            self.lbLeftDay.append(day_label)

            form.addRow("예상 복용일",day_label)

            card.setLayout(form)

            self.pill_cards.append(card)

        # ==========================
        # 레이아웃
        # ==========================

        left_layout = QVBoxLayout()

        left_layout.addWidget(self.image_label)
        left_layout.addWidget(self.btn_start)
        right_layout = QVBoxLayout()
        right_layout.setSpacing(15)

        for card in self.pill_cards:
            right_layout.addWidget(card)

        main_layout = QHBoxLayout()
        main_layout.setSpacing(20)
        main_layout.addLayout(left_layout,3)
        main_layout.addLayout(right_layout,1)

        self.setLayout(main_layout)

        # 웹캠 자동 실행
        if self.cap is None or not self.cap.isOpened():
            self.cap = cv2.VideoCapture(0)
            if not self.timer.isActive():
                self.timer.start(30)

    def start_webcam(self):
        temp = copy.deepcopy(self.pill_class)

        for i in range(3):

            pNum = 0
            edt = self.edtPdaily[i]

            if edt.text() != "":
                try:
                    pNum = int(edt.text())

                except ValueError:
                    pNum = 0

            # 현재 개수 표시
            self.lbTotalCount[i].setText(f"{temp[i]} 개")

            # 예상 복용일 계산
            if pNum != 0:
                left_day = temp[i] / pNum
                self.lbLeftDay[i].setText(f"{left_day:.1f} 일")

            else:
                self.lbLeftDay[i].setText("0 일")

    def update_frame(self):
        """
        OpenCV 프레임 처리
        YOLO 추론 로직 유지
        """
        self.pill_class = [0,0,0]

        if self.cap and self.cap.isOpened():
            ret, frame = self.cap.read()
            if ret:
                # =====================
                # 기존 이미지 전처리 유지
                # =====================
                dark = cv2.convertScaleAbs(frame,alpha=0.9,beta=-40)
                gray_frame = cv2.cvtColor(dark,cv2.COLOR_BGR2GRAY)

                # =====================
                # YOLO 추론
                # =====================

                results = self.model(gray_frame,conf=0.7,verbose=False)

                for result in results:
                    for box in result.boxes:
                        # 좌표
                        x1, y1, x2, y2 = map(int,box.xyxy[0])

                        # confidence
                        conf = float(box.conf[0])

                        # class
                        cls = int(box.cls[0])
                        if 0 <= cls <= 2:
                            self.pill_class[cls] += 1

                        name = self.model.names[cls]
                        color = self.colors.get(cls,(255,255,255))

                        # 검출 박스
                        cv2.rectangle(frame,(x1,y1),(x2,y2),color,2)

                        # 클래스 표시
                        cv2.putText(
                            frame,
                            f"{name} {conf:.2f}",
                            (x1,y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.7,
                            color,
                            2
                        )

                # =====================
                # QLabel 출력
                # =====================
                rgb_image = cv2.cvtColor(
                    frame,
                    cv2.COLOR_BGR2RGB
                )
                h,w,ch = rgb_image.shape
                bytes_per_line = ch*w
                qt_image = QImage(
                    rgb_image.data,
                    w,
                    h,
                    bytes_per_line,
                    QImage.Format.Format_RGB888
                )
                pixmap = QPixmap.fromImage(
                    qt_image
                )
                scaled_pixmap = pixmap.scaled(
                    self.image_label.size(),
                    Qt.AspectRatioMode.KeepAspectRatio,
                    Qt.TransformationMode.SmoothTransformation
                )
                self.image_label.setPixmap(
                    scaled_pixmap
                )

    def closeEvent(self, event):
        """
        프로그램 종료 시
        웹캠 자원 해제
        """
        self.timer.stop()
        if self.cap and self.cap.isOpened():
            self.cap.release()
        event.accept()

if __name__ == "__main__":

    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    exam = My_App()
    exam.show()
    sys.exit(
        app.exec()
    )

SystemExit: 0

In [ ]:
import sys

from PySide6.QtWidgets import QApplication

from ui import My_App




def main():

    app = QApplication(sys.argv)

    window = My_App()

    window.show()

    sys.exit(
        app.exec()
    )



if __name__ == "__main__":

    main()

In [ ]:
import os
import json

# JSON 파일들이 있는 폴더
folder = r"C:\Users\kccistc\Desktop\workspace\test"   # ← 여기를 수정

# 폴더 내 모든 파일 순회
for filename in os.listdir(folder):
    if not filename.lower().endswith(".json"):
        continue

    json_path = os.path.join(folder, filename)
    txt_path = os.path.join(folder, os.path.splitext(filename)[0] + ".txt")

    try:
        # JSON 읽기
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # TXT 저장
        with open(txt_path, "w", encoding="utf-8") as f:
            for obj in data:
                f.write(
                    f"{obj['class_id']} "
                    f"{obj['center_x']} "
                    f"{obj['center_y']} "
                    f"{obj['width']} "
                    f"{obj['height']}\n"
                )

        print(f"변환 완료 : {filename}")

    except Exception as e:
        print(f"오류 발생 : {filename} -> {e}")

print("모든 변환이 완료되었습니다.")